# Model Deployment of the Supervised baseline and SimCLR model

`This is the notebook where we deploy the best two of our five appraoches for further early exit experiment`

## Sections

1. SimCLR export
2. Supervised Baseline export

# SimCLR Model Export
Exports the SimCLR encoder as TorchScript for deployment, quantization, and early-exit experiments.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('Device:', DEVICE)

Device: mps


In [ ]:
# recreate SimCLR architecture
class SimCLR(nn.Module):
    def __init__(self, proj_dim=128):
        super().__init__()
        base = models.resnet18(weights=None)
        self.encoder   = nn.Sequential(*list(base.children())[:-1])  # [B, 512, 1, 1]
        self.feat_dim  = base.fc.in_features  # 512
        self.projector = nn.Sequential(
            nn.Linear(self.feat_dim, self.feat_dim),
            nn.ReLU(),
            nn.Linear(self.feat_dim, proj_dim),
        )

    def forward(self, x):
        h = self.encoder(x).squeeze(-1).squeeze(-1)  # [B, 512]
        z = self.projector(h)
        return h, F.normalize(z, dim=1)


model = SimCLR(proj_dim=128)
model.load_state_dict(torch.load('simclr_best.pt', map_location='cpu'))
model.eval()
print('SimCLR loaded.')

SimCLR loaded.


In [ ]:
# export encoder as TorchScript
# Use torch.jit.trace — more compatible than script for nn.Sequential

encoder = model.encoder

dummy = torch.randn(1, 3, 224, 224)
traced_encoder = torch.jit.trace(encoder, dummy)
traced_encoder.save('simclr_encoder_scripted.pt')

# Verify
loaded = torch.jit.load('simclr_encoder_scripted.pt')
out = loaded(dummy)  # [1, 512, 1, 1]
print('TorchScript encoder output shape:', out.shape)
print('Saved -> simclr_encoder_scripted.pt')

TorchScript encoder output shape: torch.Size([1, 512, 1, 1])
Saved -> simclr_encoder_scripted.pt


In [ ]:
# Sanity check: features from TorchScript vs original should be identical

with torch.no_grad():
    h_orig, _ = model(dummy)
    h_ts = loaded(dummy).squeeze(-1).squeeze(-1)

diff = (h_orig - h_ts).abs().max().item()
print(f'Max feature difference (orig vs TorchScript): {diff:.2e}')  # should be ~0

Max feature difference (orig vs TorchScript): 0.00e+00


## Supervised Model Export
Exports the supervised ResNet18 backbone (without fc head) as TorchScript.

In [ ]:
# load supervised model and strip fc head
from torchvision import models

NUM_CLASSES = 2

sup_full = models.resnet18(weights=None)
sup_full.fc = torch.nn.Linear(512, NUM_CLASSES)
sup_full.load_state_dict(torch.load("sup_best.pt", map_location="cpu"))
sup_full.eval()

# Replace fc with Identity to get backbone-only output (512-dim)
sup_backbone = sup_full
sup_backbone.fc = torch.nn.Identity()
sup_backbone.eval()

print("Supervised backbone loaded.")
dummy = torch.randn(1, 3, 224, 224)
out = sup_backbone(dummy)
print("Backbone output shape:", out.shape)  # [1, 512]

Supervised backbone loaded.
Backbone output shape: torch.Size([1, 512])


In [ ]:
# export supervised backbone as TorchScript
traced_sup = torch.jit.trace(sup_backbone, dummy)
traced_sup.save("sup_backbone_scripted.pt")

# Verify 
loaded_sup = torch.jit.load("sup_backbone_scripted.pt")
out_ts = loaded_sup(dummy)
diff = (out - out_ts).abs().max().item()
print(f"Max feature difference (orig vs TorchScript): {diff:.2e}")
print("Saved -> sup_backbone_scripted.pt")

Max feature difference (orig vs TorchScript): 0.00e+00
Saved -> sup_backbone_scripted.pt
